# Phase 4 — Performance Evaluation

## Goal
Evaluate the fine-tuned BART model using ROUGE metrics and visualize the training loss curve.

## Why ROUGE and not Accuracy/Confusion Matrix?
This is a **text generation** task, not classification.
- Confusion Matrix → requires fixed classes ❌
- Accuracy → meaningless for generated text ❌
- **ROUGE** → measures overlap between generated and reference summaries ✅

## ROUGE Metrics
| Metric | Measures |
|---|---|
| ROUGE-1 | Overlap of single words (unigrams) |
| ROUGE-2 | Overlap of word pairs (bigrams) |
| ROUGE-L | Longest common subsequence |

In [ ]:
import os
os.environ["HF_HOME"] = r"D:\hf_cache"

from datasets import load_from_disk
from transformers import AutoTokenizer, BartForConditionalGeneration
import evaluate
import matplotlib.pyplot as plt
import json

## 1. Load Model & Data

In [ ]:
MODEL_PATH = "../outputs/model"
tokenizer  = AutoTokenizer.from_pretrained("facebook/bart-base")
model      = BartForConditionalGeneration.from_pretrained(MODEL_PATH)
model.eval()

test_data = load_from_disk("../data/processed/test")
print(f"Test examples: {len(test_data)}")

## 2. Generate Summaries

`model.generate()` produces the summary using **Beam Search**:
- Explores 4 candidate sequences simultaneously
- Picks the best one at the end
- Better quality than Greedy Search (which picks the first best word each time)

In [ ]:
def generate_summary(text):
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )
    summary_ids = model.generate(
        inputs["input_ids"],
        max_new_tokens=128,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

predictions = []
references  = []

for example in test_data:
    pred = generate_summary(example["document"])
    predictions.append(pred)
    references.append(example["summary"])

print(f"Generated {len(predictions)} summaries!")

## 3. ROUGE Scores

In [ ]:
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=predictions, references=references)

print("--- ROUGE Scores ---")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

os.makedirs("../outputs/reports", exist_ok=True)
with open("../outputs/reports/rouge_scores.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nScores saved to outputs/reports/rouge_scores.json")

## 4. Training Loss Curve

A **decreasing loss** means the model is learning correctly.
If loss increases again → overfitting (model memorizing instead of learning).

In [ ]:
log_file = "../outputs/model/trainer_state.json"

with open(log_file) as f:
    trainer_state = json.load(f)

steps  = [x["step"] for x in trainer_state["log_history"] if "loss" in x]
losses = [x["loss"] for x in trainer_state["log_history"] if "loss" in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, label="Training Loss", color="blue")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("../outputs/plots/loss_curve.png")
plt.show()
print("Loss curve saved!")

## 5. Sample Predictions

In [ ]:
print("--- Sample Predictions ---")
for i in range(3):
    print(f"\nExample {i+1}:")
    print(f"Reference:  {references[i]}")
    print(f"Generated:  {predictions[i]}")

## Summary of Results

| Metric | Score |
|---|---|
| ROUGE-1 | 0.3324 |
| ROUGE-2 | 0.1284 |
| ROUGE-L | 0.2716 |

The model successfully learned to generate coherent news summaries after fine-tuning on the XSum dataset.
The decreasing training loss confirms that the model improved across epochs.